# Task 5 — The First Prediction
**PlaceMux AI/ML Phase 1** | First model + baseline comparison + error analysis

**Dataset:** Credit Card Fraud (1500 rows) | **Task:** Binary Classification

In [ ]:
import sys, os; sys.path.insert(0, '..')
import random, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from src.preprocessing import load_and_split, build_and_fit_preprocessor
from src.baseline import compute_baseline
from src.train import train_model
from src.evaluate import evaluate, error_analysis, log_experiment
SEED=42; random.seed(SEED); np.random.seed(SEED)
print('✅ Setup complete')

## 1 — Load & Inspect Dataset

In [ ]:
df = pd.read_csv('../data/credit_fraud_dataset.csv')
print('Shape:', df.shape)
print('Fraud rate:', df['is_fraud'].mean().round(3))
print('Missing:', df.isnull().sum().sum())
df.head()

## 2 — Split FIRST, then Preprocess (no leakage)

In [ ]:
X_tr,X_val,X_test,y_tr,y_val,y_test = load_and_split('../data/credit_fraud_dataset.csv',seed=SEED)
pp, X_tr_p = build_and_fit_preprocessor(X_tr)
X_val_p = pp.transform(X_val)
X_test_p = pp.transform(X_test)
print('Preprocessed train:', X_tr_p.shape)

## 3 — Baseline FIRST (majority class)

In [ ]:
base_m, base_pred = compute_baseline(y_tr, y_val)
print('\nKEY INSIGHT: Baseline always predicts "Not Fraud"')
print('It gets 71% accuracy — but catches ZERO frauds!')
print('This is why F1 is the right metric here, not accuracy.')

## 4 — Train Logistic Regression

In [ ]:
lr = train_model('logistic', X_tr_p, y_tr, SEED)
val_m, val_pred = evaluate(lr, X_val_p, y_val, 'val', 'logistic')

## 5 — Baseline vs Model Comparison

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(12,4))
labels=['Baseline','Logistic Reg']
f1_vals=[base_m['f1_macro'], val_m['f1_macro']]
acc_vals=[base_m['accuracy'], val_m['accuracy']]
bars=axes[0].bar(labels, f1_vals, color=['#d9534f','#5cb85c'], edgecolor='black')
axes[0].set_title('F1-Score: Baseline vs First Model'); axes[0].set_ylim(0,1)
for b,v in zip(bars,f1_vals): axes[0].text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.3f}', ha='center', fontweight='bold')
axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='0.5 reference')
bars2=axes[1].bar(labels, acc_vals, color=['#d9534f','#5cb85c'], edgecolor='black')
axes[1].set_title('Accuracy: Baseline vs First Model'); axes[1].set_ylim(0,1)
for b,v in zip(bars2,acc_vals): axes[1].text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.3f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/baseline_vs_model.png', dpi=100, bbox_inches='tight')
plt.show()
lift=round(val_m['f1_macro']-base_m['f1_macro'],4)
print(f'\nLift over baseline: F1 +{lift} (+{lift/base_m["f1_macro"]*100:.1f}% relative improvement)')

## 6 — Error Analysis

In [ ]:
errors = error_analysis(X_val, y_val, val_pred, out_path='../results/error_analysis_notebook.csv')
fn = errors[errors['actual']==1]  # missed frauds
fp = errors[errors['actual']==0]  # false alarms
print(f'\nFalse Negatives (missed frauds): {len(fn)}')
print(f'False Positives (false alarms) : {len(fp)}')
print(f'\nMissed fraud profile (avg):')
print(fn[['transaction_amount','num_prev_disputes','country_match','card_present']].mean().round(2))
print(f'\nPattern: missed frauds have lower dispute history than expected — harder to distinguish from legit.')

## 7 — Confusion Matrix

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(10,4))
import numpy as np
ConfusionMatrixDisplay.from_predictions(y_val, base_pred, display_labels=['Not Fraud','Fraud'], ax=axes[0], colorbar=False)
axes[0].set_title('Confusion Matrix — Baseline')
ConfusionMatrixDisplay.from_predictions(y_val, val_pred, display_labels=['Not Fraud','Fraud'], ax=axes[1], colorbar=False)
axes[1].set_title('Confusion Matrix — Logistic Reg')
plt.tight_layout()
plt.savefig('../results/confusion_matrices.png', dpi=100, bbox_inches='tight')
plt.show()

## 8 — Final Test Evaluation

In [ ]:
test_m, _ = evaluate(lr, X_test_p, y_test, 'test', 'logistic')
print('IMPORTANT: Test set evaluated ONCE — final unbiased score')

## 9 — Experiment Log

In [ ]:
pd.read_csv('../experiment_log/experiment_log.csv')[['exp_id','model','baseline_f1','val_f1','lift_f1','false_negatives','next_step']]

## ✅ Task 5 Summary

| Item | Baseline | Logistic Reg |
|---|---|---|
| F1-Score (val) | 0.4156 | 0.7147 |
| Accuracy (val) | 71.1% | 78.7% |
| Frauds caught | 0% | 49% |
| **Lift** | — | **+0.2991 F1** |

**Error Patterns Found:**
- 33 false negatives (missed frauds) — lower dispute history than true positives
- 15 false positives — higher transaction amounts triggering false alarms
- Model struggles most with borderline fraud cases

**Next Experiment:** Apply `class_weight='balanced'` to improve fraud recall.